# Thoracic Disease Detection — Full Colab Run

This notebook stages NIH ChestXray14 onto Colab's local disk, trains DenseNet-121 and ViT-B/16, evaluates both models, and generates Grad-CAM outputs. Select a GPU runtime before starting.

In [ ]:
import torch

print('PyTorch:', torch.__version__)
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))

In [ ]:
from google.colab import drive

drive.mount('/content/drive')

In [ ]:
import os
import subprocess
from pathlib import Path

REPO_URL = 'https://github.com/sethu-ram-reddy/thoracic-disease-detection.git'
REPO_DIR = Path('/content/thoracic-disease-detection')

if REPO_DIR.exists():
    subprocess.run(['git', '-C', str(REPO_DIR), 'pull', '--ff-only'], check=True)
else:
    subprocess.run(['git', 'clone', REPO_URL, str(REPO_DIR)], check=True)
os.chdir(REPO_DIR)
print('Working directory:', Path.cwd())

In [ ]:
subprocess.run(['pip', 'install', '-q', '-r', 'requirements-colab.txt'], check=True)
subprocess.run(['pip', 'install', '-q', '-e', '.'], check=True)

## Stage the dataset locally

The previous notebook trained directly from mounted Drive. Copying the dataset to `/content` removes that input bottleneck. The copy is required again after a fresh Colab runtime.

In [ ]:
DRIVE_DATA_DIR = Path('/content/drive/MyDrive/Interpretable_DL_Medical_Imaging/data')
LOCAL_DATA_DIR = Path('/content/data/nih-chest-xray')
LOCAL_DATA_DIR.mkdir(parents=True, exist_ok=True)

if not DRIVE_DATA_DIR.exists():
    raise FileNotFoundError(f'Dataset folder not found: {DRIVE_DATA_DIR}')
subprocess.run(
    ['rsync', '-a', '--info=progress2', f'{DRIVE_DATA_DIR}/', f'{LOCAL_DATA_DIR}/'],
    check=True,
)

In [ ]:
subprocess.run(
    ['python', 'scripts/audit_dataset.py', '--data-root', str(LOCAL_DATA_DIR)],
    check=True,
)

## DenseNet-121

In [ ]:
subprocess.run(
    ['python', 'scripts/train.py', '--config', 'configs/densenet121.yaml', '--data-root', str(LOCAL_DATA_DIR)],
    check=True,
)

In [ ]:
subprocess.run(
    ['python', 'scripts/evaluate.py', '--config', 'configs/densenet121.yaml', '--data-root', str(LOCAL_DATA_DIR)],
    check=True,
)
subprocess.run(
    ['python', 'scripts/generate_gradcam.py', '--config', 'configs/densenet121.yaml', '--data-root', str(LOCAL_DATA_DIR), '--samples', '12'],
    check=True,
)

## Vision Transformer

In [ ]:
subprocess.run(
    ['python', 'scripts/train.py', '--config', 'configs/vit_b16.yaml', '--data-root', str(LOCAL_DATA_DIR)],
    check=True,
)
subprocess.run(
    ['python', 'scripts/evaluate.py', '--config', 'configs/vit_b16.yaml', '--data-root', str(LOCAL_DATA_DIR)],
    check=True,
)

## Model comparison

In [ ]:
DENSENET_DIR = '/content/drive/MyDrive/Thoracic_Disease_Detection/outputs/densenet121'
VIT_DIR = '/content/drive/MyDrive/Thoracic_Disease_Detection/outputs/vit_b16'
COMPARISON_DIR = '/content/drive/MyDrive/Thoracic_Disease_Detection/results'

subprocess.run(
    [
        'python', 'scripts/compare_models.py',
        '--densenet-dir', DENSENET_DIR,
        '--vit-dir', VIT_DIR,
        '--output-dir', COMPARISON_DIR,
    ],
    check=True,
)